# FightWatch — đối chứng pretrained đấm/đá
So sánh checkpoint đang dùng với AVA Slow R50 và SlowFast R50 chính thức. Không training, không tự thay model; video iPhone không tải lên. Chỉ lấy 16 video public đã định danh SHA, tách calibration/development/test theo người biểu diễn. Báo cáo riêng đấm/đá/ôm/bắt tay. Đây là chẩn đoán public nhỏ, không phải nghiệm thu camera thật. Tự ngắt GPU sau khi xong hoặc tối đa 45 phút.

In [ ]:
import time, threading
from pathlib import Path
import torch
from google.colab import drive, runtime
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > GPU trước.'
if 'ACTION_DEADLINE' not in globals():
    ACTION_DEADLINE = time.monotonic() + 45*60
    watchdog = threading.Timer(45*60, runtime.unassign)
    watchdog.daemon = True
    watchdog.start()
drive.mount('/content/drive', timeout_ms=180000)
ROOT = Path('/content/drive/MyDrive/SENTINEL')
REPO = ROOT/'repo/anomaly-behavior-detector-system'
OUTPUT = ROOT/'artifacts/fight_punch_kick_pretrained_v1'
LOCAL = Path('/content/fightwatch-action-code')
OUTPUT.mkdir(parents=True, exist_ok=True)
def remaining(reserve=0):
    seconds = ACTION_DEADLINE-time.monotonic()-reserve
    if seconds <= 0:
        raise TimeoutError('Hết thời gian; log đã lưu trên Drive.')
    return seconds
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os, sys, json, subprocess, urllib.request, zipfile, shutil
try:
    result = subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True, timeout=min(90,remaining()), capture_output=True, text=True)
    REVISION = subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD'],text=True,timeout=20).strip()
    assert len(REVISION)==40 and all(char in '0123456789abcdef' for char in REVISION)
    archive = Path('/content/fightwatch-action-code.zip')
    with urllib.request.urlopen('https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/'+REVISION, timeout=90) as response, archive.open('wb') as handle:
        shutil.copyfileobj(response,handle)
    LOCAL.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as package:
        for item in package.infolist():
            target = (LOCAL/item.filename).resolve()
            assert target.is_relative_to(LOCAL.resolve()), 'Unsafe source archive'
        package.extractall(LOCAL)
    CODE = LOCAL/('anomaly-behavior-detector-system-'+REVISION)
    os.chdir(CODE)
    sys.path.insert(0,str(CODE))
    from scripts.colab_process import run_logged
    run_logged([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],OUTPUT/'install.log',min(600,remaining(180)),revision=REVISION)
    print('Code:',REVISION, 'Repo trên Drive không bị ghi đè.')
except BaseException:
    runtime.unassign()
    raise


In [ ]:
try:
    models = Path('/content/action-models')
    models.mkdir(exist_ok=True)
    bundle = ROOT/'artifacts/fight_fusion_v1/rgb_aux/sentinel_fight_only_runtime.zip'
    assert bundle.is_file(), 'Không tìm thấy gói model fusion trên Drive'
    with zipfile.ZipFile(bundle) as package:
        for name in ('fight_model.ts','fight_runtime.json','yolov8n-pose.pt'):
            matches = [item for item in package.infolist() if Path(item.filename).name==name and not item.is_dir()]
            assert len(matches)==1, 'Model missing/ambiguous: '+name
            with package.open(matches[0]) as source, (models/name).open('wb') as target:
                shutil.copyfileobj(source,target)
    pku = ROOT/'datasets/fight (1)/pku_mmd_sample/rgb/0360-R.avi'
    assert pku.is_file(), 'Thiếu RGB PKU 0360-R tại đường dẫn đã dùng trước đây'
    weights = OUTPUT/'public_pretrained_weights'
    weights.mkdir(exist_ok=True)
    minutes = min(30,remaining(180)/60)
    assert minutes >= 1, 'Không còn đủ thời gian để chạy'
    command = [sys.executable,'-m','scripts.fight_action_compare',
        '--corpus',str(CODE/'benchmarks/punch_kick_v1.json'),
        '--models',str(models),'--pku',str(pku),
        '--data','/content/action-public-data','--weights',str(weights),
        '--output',str(OUTPUT),'--device','cuda','--minutes',str(minutes)]
    run_logged(command, OUTPUT/'compare.log', min(minutes*60+30,remaining(120)), revision=REVISION)
    report = json.loads((OUTPUT/'run.json').read_text())
    print(json.dumps(report,ensure_ascii=False,indent=2))
    print('Báo cáo:',OUTPUT,'Không thay checkpoint local; chưa nghiệm thu iPhone thật.')
finally:
    print('Ngắt GPU; báo cáo/các trace đã lưu trên Drive.',flush=True)
    runtime.unassign()
